# Separate validation — unseen scattering coefficients
Evaluate frozen DRS-only LOSO checkpoints directly from `NIRS_Absolute_Val_Dataset_new.mat`. DRS rows are loaded and preprocessed in batches; the same workflow supports HC and StO₂.

## Parameters
Set `TARGET` to `'hc'` or `'sto2'`. Checkpoint paths, outputs, plot labels, and the cache target column are selected automatically.

In [8]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():  # VS Code may start in notebooks/
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from final_refactored or its notebooks folder.')
os.chdir(ROOT)

SRC_DIR = ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + SUBPROCESS_ENV.get('PYTHONPATH', '')

TARGET = 'sto2'  # 'hc' or 'sto2'
FOLDS = None   # None for all folds, or e.g. '1,3,10-20'
DEVICE = 'cuda:1'  # 'auto', 'cpu', 'cuda', or e.g. 'cuda:1'
BATCH_SIZE = 4096
SAVE_PREDICTIONS = False

if TARGET not in {'hc', 'sto2'}:
    raise ValueError("TARGET must be 'hc' or 'sto2'")
CHECKPOINT_DIR = Path(f'artifacts/stage2/{TARGET}_baseline_scratch_main')
VALIDATION_MAT = Path('data/NIRS_Absolute_Val_Dataset_new.mat')
OUTPUT_DIR = Path(f'artifacts/stage2/unseen_scattering/{TARGET}')
FIGURE_DIR = Path('results/figures/stage2/unseen_scattering')
UNIT = 'µM' if TARGET == 'hc' else 'fraction'
TITLE = f'{TARGET.upper()} prediction errors under unseen scattering spectra'

TARGET, CHECKPOINT_DIR, VALIDATION_MAT, OUTPUT_DIR

('sto2',
 PosixPath('artifacts/stage2/sto2_baseline_scratch_main'),
 PosixPath('data/NIRS_Absolute_Val_Dataset_new.mat'),
 PosixPath('artifacts/stage2/unseen_scattering/sto2'))

## 1. Evaluate the selected baseline
Each fold evaluates only its LOSO test subject. Fold JSON files make the run resumable.

In [9]:
if not VALIDATION_MAT.is_file():
    raise FileNotFoundError(f'Set VALIDATION_MAT to NIRS_Absolute_Val_Dataset_new.mat. Missing: {VALIDATION_MAT}')

command = [
    sys.executable, '-m', 'subject_nirs.stage2.unseen_scattering',
    '--target', TARGET,
    '--checkpoint_dir', str(CHECKPOINT_DIR),
    '--validation_mat', str(VALIDATION_MAT),
    '--output_dir', str(OUTPUT_DIR),
    '--device', DEVICE,
    '--batch_size', str(BATCH_SIZE),
]
if FOLDS:
    command.extend(['--folds', FOLDS])
if not SAVE_PREDICTIONS:
    command.append('--no_save_predictions')
subprocess.run(command, cwd=ROOT, env=SUBPROCESS_ENV, check=True)

target=sto2 device=cuda:1 folds=154 source=mat rows=7,983,360 output=artifacts/stage2/unseen_scattering/sto2
[001/154] fold 001: rows=51,840 MAE=0.032508 bias=-0.027625 sec=2.2
[002/154] fold 002: rows=51,840 MAE=0.047750 bias=-0.044495 sec=0.1
[003/154] fold 003: rows=51,840 MAE=0.044115 bias=0.027839 sec=0.1
[004/154] fold 004: rows=51,840 MAE=0.043823 bias=-0.021507 sec=0.1
[005/154] fold 005: rows=51,840 MAE=0.097616 bias=0.084142 sec=0.1
[006/154] fold 006: rows=51,840 MAE=0.031271 bias=-0.012455 sec=0.1
[007/154] fold 007: rows=51,840 MAE=0.031876 bias=-0.017629 sec=0.1
[008/154] fold 008: rows=51,840 MAE=0.035799 bias=-0.022604 sec=0.1
[009/154] fold 009: rows=51,840 MAE=0.039366 bias=-0.004624 sec=0.1
[010/154] fold 010: rows=51,840 MAE=0.041440 bias=-0.013135 sec=0.1
[011/154] fold 011: rows=51,840 MAE=0.030485 bias=0.020988 sec=0.1
[012/154] fold 012: rows=51,840 MAE=0.046125 bias=0.040018 sec=0.1
[013/154] fold 013: rows=51,840 MAE=0.055625 bias=-0.013684 sec=0.1
[014/154] f

CompletedProcess(args=['/home/md703/.conda/envs/yc_ae/bin/python', '-m', 'subject_nirs.stage2.unseen_scattering', '--target', 'sto2', '--checkpoint_dir', 'artifacts/stage2/sto2_baseline_scratch_main', '--validation_mat', 'data/NIRS_Absolute_Val_Dataset_new.mat', '--output_dir', 'artifacts/stage2/unseen_scattering/sto2', '--device', 'cuda:1', '--batch_size', '4096', '--no_save_predictions'], returncode=0)

## 2. Plot fold distributions (PNG only)

In [10]:
METRICS_CSV = OUTPUT_DIR / 'per_fold_metrics.csv'
if not METRICS_CSV.is_file():
    raise FileNotFoundError(f'Evaluation did not create {METRICS_CSV}')
subprocess.run(
    [
        sys.executable, '-m', 'subject_nirs.stage2.unseen_scattering_plot',
        '--input_csv', str(METRICS_CSV),
        '--output_dir', str(FIGURE_DIR),
        '--output_prefix', f'fig_unseen_scattering_{TARGET}_rmse_bias_distribution',
        '--unit', UNIT,
        '--title', TITLE,
    ],
    cwd=ROOT, env=SUBPROCESS_ENV, check=True,
)

RMSE: mean±SD=0.057±0.023
Bias: mean±SD=-0.004±0.036
saved: results/figures/stage2/unseen_scattering/fig_unseen_scattering_sto2_rmse_bias_distribution.png
saved: results/figures/stage2/unseen_scattering/fig_unseen_scattering_sto2_rmse_bias_distribution_metadata.json


CompletedProcess(args=['/home/md703/.conda/envs/yc_ae/bin/python', '-m', 'subject_nirs.stage2.unseen_scattering_plot', '--input_csv', 'artifacts/stage2/unseen_scattering/sto2/per_fold_metrics.csv', '--output_dir', 'results/figures/stage2/unseen_scattering', '--output_prefix', 'fig_unseen_scattering_sto2_rmse_bias_distribution', '--unit', 'fraction', '--title', 'STO2 prediction errors under unseen scattering spectra'], returncode=0)